# Week 3 — Needleman–Wunsch playground

Open in VS Code. **Select Kernel** (Python 3). Run from the top.

Earlier: tourist fill. Now: **three moves** and a score $\sigma$ on letters (same meeting).
Change the knobs, watch the alignment change.


## Scoring knobs

Lecture toy: match $+1$, mismatch $-1$, gap $-1$. Sink score should be $2$.


In [ ]:
# --- change these and re-run ---
MATCH = 1
MISMATCH = -1
GAP = 1          # cost; we subtract GAP
v = "GCT"
w = "GCAT"
# --------------------------------

def sigma(a, b):
    return MATCH if a == b else MISMATCH

print(f"sigma: match={MATCH}, mismatch={MISMATCH}, gap cost={GAP}")
print(f"align {v} vs {w}")


## Needleman–Wunsch with traceback

Store pointers `D`: `"D"` diagonal, `"U"` up (gap in $w$), `"L"` left (gap in $v$).


In [ ]:
def nw(v, w, sigma, gap=GAP):
    n, m = len(v), len(w)
    S = [[0] * (m + 1) for _ in range(n + 1)]
    D = [[None] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        S[i][0] = -i * gap
        D[i][0] = "U"
    for j in range(1, m + 1):
        S[0][j] = -j * gap
        D[0][j] = "L"
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            diag = S[i - 1][j - 1] + sigma(v[i - 1], w[j - 1])
            up = S[i - 1][j] - gap
            left = S[i][j - 1] - gap
            best = max(diag, up, left)
            S[i][j] = best
            if best == diag:
                D[i][j] = "D"
            elif best == up:
                D[i][j] = "U"
            else:
                D[i][j] = "L"
    return S, D

def traceback_nw(v, w, D):
    i, j = len(v), len(w)
    top, bot = [], []
    while i > 0 or j > 0:
        move = D[i][j]
        if move == "D":
            top.append(v[i - 1]); bot.append(w[j - 1]); i -= 1; j -= 1
        elif move == "U":
            top.append(v[i - 1]); bot.append("-"); i -= 1
        else:
            top.append("-"); bot.append(w[j - 1]); j -= 1
    return "".join(reversed(top)), "".join(reversed(bot))

S, D = nw(v, w, sigma)
aln_v, aln_w = traceback_nw(v, w, D)
print("S:")
for row in S:
    print(row)
print(f"sink S[{len(v)}][{len(w)}] = {S[len(v)][len(w)]}")
print(aln_v)
print(aln_w)
assert S[3][4] == 2
assert aln_v.replace(" ", "") == "GC-T" or aln_v == "GC-T"
# lecture optimum
assert (aln_v, aln_w) == ("GC-T", "GCAT")
print("matches lecture alignment GC-T / GCAT")


## Play with scores

Set `MISMATCH = -3` or `GAP = 2` above and re-run. Does the alignment change?


In [ ]:
# Re-run with current knobs (edit MATCH/MISMATCH/GAP in the setup cell first)
S2, D2 = nw(v, w, sigma)
a2, b2 = traceback_nw(v, w, D2)
print(f"score {S2[len(v)][len(w)]}")
print(a2)
print(b2)


## Local vs global

Flanking junk: global pays gap/mismatch tax everywhere; local finds the shared stretch.


In [ ]:
def sw(v, w, sigma, gap=GAP):
    n, m = len(v), len(w)
    S = [[0] * (m + 1) for _ in range(n + 1)]
    D = [[None] * (m + 1) for _ in range(n + 1)]
    best_score, best_ij = 0, (0, 0)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            diag = S[i - 1][j - 1] + sigma(v[i - 1], w[j - 1])
            up = S[i - 1][j] - gap
            left = S[i][j - 1] - gap
            best = max(0, diag, up, left)
            S[i][j] = best
            if best == 0:
                D[i][j] = None
            elif best == diag:
                D[i][j] = "D"
            elif best == up:
                D[i][j] = "U"
            else:
                D[i][j] = "L"
            if best > best_score:
                best_score, best_ij = best, (i, j)
    return S, D, best_score, best_ij

def traceback_sw(v, w, S, D, start):
    i, j = start
    top, bot = [], []
    while i > 0 and j > 0 and D[i][j] is not None and S[i][j] > 0:
        move = D[i][j]
        if move == "D":
            top.append(v[i - 1]); bot.append(w[j - 1]); i -= 1; j -= 1
        elif move == "U":
            top.append(v[i - 1]); bot.append("-"); i -= 1
        else:
            top.append("-"); bot.append(w[j - 1]); j -= 1
    return "".join(reversed(top)), "".join(reversed(bot))

v_long = "XXXGCTXXX"
w_long = "YYYGCATYYY"
S_g, D_g = nw(v_long, w_long, sigma)
S_sw, D_sw, loc_score, loc_ij = sw(v_long, w_long, sigma)
loc_a, loc_b = traceback_sw(v_long, w_long, S_sw, D_sw, loc_ij)

print(f"NW global sink = {S_g[len(v_long)][len(w_long)]}")
ga, gb = traceback_nw(v_long, w_long, D_g)
print(ga)
print(gb)
print(f"\nSW local max = {loc_score} at {loc_ij}")
print(loc_a)
print(loc_b)



## Optional — “biological” $\sigma$

Leu/Ile-like: similar letters less punished. Same NW loop.


In [ ]:
# toy alphabet with a friendly pair
friendly = {("L", "I"), ("I", "L")}

def sigma_bio(a, b):
    if a == b:
        return 4
    if (a, b) in friendly:
        return 2   # similar
    return -1

# DNA demo: treat A-G as a mild "transition"
def sigma_dna(a, b):
    if a == b:
        return 1
    if frozenset((a, b)) in (frozenset(("A", "G")), frozenset(("C", "T"))):
        return 0   # transition
    return -2      # transversion

for name, sig in [("uniform ±1", sigma), ("DNA ti/tv", sigma_dna)]:
    Sx, Dx = nw("GCT", "GCAT", sig)
    ax, bx = traceback_nw("GCT", "GCAT", Dx)
    print(f"{name:12s}  score={Sx[3][4]:3d}  {ax} / {bx}")

# protein-ish pair
Sp, Dp = nw("ALI", "AII", sigma_bio)
ap, bp = traceback_nw("ALI", "AII", Dp)
print(f"{'BLOSUM-toy':12s}  score={Sp[3][3]:3d}  {ap} / {bp}")


## Questions

**Q1.** The three moves, in words.

A1.

**Q2.** Why is $T(n,m)=O(nm)$? What does changing $\sigma$ change — the algorithm, or the answer?

A2.

**Q3.** The $n\times m$ table does not fit in RAM. What does Hirschberg split, and which two numbers do you add at the cut?

A3.
